# Azeitona e azeite em Portugal: análise exploratória inicial
**Estágio de Iniciação Científica 2026/27 – S1 (UPB)** · tarefa T2 (limpeza, harmonização e exploração)

O que este notebook faz:
1. Lê os 4 CSVs exportados do INE (**não precisa renomear**: ele reconhece cada arquivo pelo título).
2. Junta tudo numa tabela única por **região agrária × ano**.
3. Faz verificações de qualidade (anos em falta, somas, consistência entre séries).
4. Faz os primeiros gráficos e a análise de alternância produtiva.
5. Deixa pronta a função de **correlação cruzada** para quando a série de oliveiras estiver disponível.

**Como usar:** coloque os CSVs numa pasta `dados_brutos/` ao lado do notebook (ou na mesma pasta dele) e rode as células em ordem.

In [ ]:
import re, glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
pd.options.display.float_format = "{:,.2f}".format

# pasta com os CSVs: tenta "dados_brutos/" e, se não existir, a pasta atual
DATA_DIR = "dados_brutos" if os.path.isdir("dados_brutos") else "."
OUT_DIR = "dados_tratados"
os.makedirs(OUT_DIR, exist_ok=True)
print("Lendo CSVs de:", os.path.abspath(DATA_DIR))

## 1. Leitura dos arquivos do INE
Os CSVs do INE têm cabeçalhos longos, notas no rodapé e dois formatos diferentes (anos nas colunas ou anos nas linhas). As funções abaixo tratam isso e devolvem uma tabela "longa": `ano, regiao, variavel, valor`.

Duas coisas dos arquivos:
- `- -` significa **sem dado** (Açores e Madeira antes de 2005 na série de azeitona).
- Alguns valores vêm com `*` (ex.: `45443 *`). O INE marca assim valores com alguma ressalva; **confirme o significado na metainformação do quadro**. Aqui o `*` é removido do número e guardado na coluna `flag_asterisco`.

In [ ]:
VARIAVEIS = {  # texto do título -> nome curto da variável (a ordem importa)
    "Azeite produzido por quintal": "rendimento_hl_q",
    "Azeite produzido (hl)":        "azeite_hl",
    "Azeitona oleificada":          "azeitona_oleificada_t",
    "Produção de azeitona":         "azeitona_producao_t",
}
RE_REGIAO = re.compile(r"^\s*(PT|\d)\s*:\s*(.+?)\s*$")

def ler_linhas(caminho):
    with open(caminho, "rb") as f:
        return f.read().decode("latin-1").splitlines()

def identificar_variavel(linhas):
    topo = " ".join(linhas[:15])
    for chave, nome in VARIAVEIS.items():
        if chave in topo:
            return nome
    return None

def limpar_valor(txt):
    txt = txt.strip()
    flag = "*" in txt
    t = txt.replace("*", "").replace(" ", "")
    if t in {"", "-", "--", "..", "x"}:
        return np.nan, flag
    try:
        return float(t.replace(",", ".")), flag
    except ValueError:
        return np.nan, flag

def _registo(ano, cod_reg, nome_reg, bruto, var):
    v, f = limpar_valor(bruto)
    cod = 0 if cod_reg == "PT" else int(cod_reg)
    return dict(ano=int(ano), regiao_cod=cod, regiao=nome_reg, variavel=var, valor=v, flag_asterisco=f)

def ler_formato_largo(linhas, var):
    """Regiões nas linhas, anos nas colunas."""
    anos, regs = None, []
    for l in linhas:
        c = l.split(";")
        if anos is None:
            ys = [x.strip() for x in c[1:] if x.strip()]
            if len(ys) > 3 and all(re.fullmatch(r"\d{4}", y) for y in ys):
                anos = ys
            continue
        m = RE_REGIAO.match(c[0])
        if m:
            for ano, bruto in zip(anos, c[1:1 + len(anos)]):
                regs.append(_registo(ano, m.group(1), m.group(2), bruto, var))
    return regs

def ler_formato_longo(linhas, var):
    """Anos nas linhas (ano;região;valor)."""
    regs, ano = [], None
    for l in linhas:
        c = l.split(";")
        if len(c) < 3:
            if regs:   # fim da tabela
                break
            continue
        if re.fullmatch(r"\d{4}", c[0].strip()):
            ano = c[0].strip()
        m = RE_REGIAO.match(c[1])
        if ano and m:
            regs.append(_registo(ano, m.group(1), m.group(2), c[2], var))
    return regs

def ler_csv_ine(caminho):
    linhas = ler_linhas(caminho)
    var = identificar_variavel(linhas)
    if var is None:
        return None
    # formato largo: a linha de cabeçalho tem vários anos separados por ";"
    largo = any(re.match(r"^;(\d{4};){5,}", l) for l in linhas[:20])
    regs = ler_formato_largo(linhas, var) if largo else ler_formato_longo(linhas, var)
    return pd.DataFrame(regs)

partes = []
for caminho in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    df_i = ler_csv_ine(caminho)
    if df_i is None or df_i.empty:
        print("  (ignorado)", os.path.basename(caminho))
        continue
    print(f"  {df_i['variavel'].iloc[0]:<24} <- {os.path.basename(caminho)}  "
          f"({df_i['ano'].min()}–{df_i['ano'].max()}, {len(df_i)} linhas)")
    partes.append(df_i)

longo = pd.concat(partes, ignore_index=True)
longo.head()

## 2. Tabela única (painel região × ano)
- **Portugal** (código 0) é o total nacional, guardado à parte.
- Usamos as **7 regiões do Continente** (códigos 1–7). Açores e Madeira ficam de fora: produzem 0 e têm vários anos sem dado.

In [ ]:
CONTINENTE = list(range(1, 8))
painel = (longo[longo.regiao_cod.isin(CONTINENTE)]
          .pivot_table(index=["ano", "regiao_cod", "regiao"], columns="variavel", values="valor")
          .reset_index())
painel.columns.name = None

nacional = (longo[longo.regiao_cod == 0]
            .pivot_table(index="ano", columns="variavel", values="valor"))

painel.to_csv(os.path.join(OUT_DIR, "painel_regiao_ano.csv"), index=False)
nacional.to_csv(os.path.join(OUT_DIR, "nacional_ano.csv"))
print("Painel:", painel.shape, "| anos:", painel.ano.min(), "a", painel.ano.max())
painel.head(8)

## 3. Verificações de qualidade
Antes de modelar, confirme que os dados estão coerentes.

In [ ]:
# 3.1 Cobertura: primeiro e último ano de cada variável
cobertura = longo.dropna(subset=["valor"]).groupby("variavel").ano.agg(["min", "max", "nunique"])
print(cobertura)

In [ ]:
# 3.2 A soma das regiões bate com o total de Portugal? (diferença relativa máxima por variável)
somas = (longo[longo.regiao_cod.between(1, 9)].groupby(["variavel", "ano"]).valor.sum(min_count=1)
         .rename("soma_regioes").reset_index()
         .merge(longo[longo.regiao_cod == 0][["variavel", "ano", "valor"]].rename(columns={"valor": "total_PT"}),
                on=["variavel", "ano"]))
# rendimento (hl/q) é uma razão e não se soma: ignorar
somas = somas[somas.variavel != "rendimento_hl_q"]
somas["dif_%"] = 100 * (somas.soma_regioes - somas.total_PT).abs() / somas.total_PT
print(somas.groupby("variavel")["dif_%"].max().round(3))

In [ ]:
# 3.3 Consistência: azeite (hl) ≈ rendimento (hl/q) × azeitona oleificada (t) × 10 (1 t = 10 q)
chk = nacional[["azeite_hl", "rendimento_hl_q", "azeitona_oleificada_t"]].dropna().copy()
chk["azeite_estimado"] = chk.rendimento_hl_q * chk.azeitona_oleificada_t * 10
chk["erro_%"] = 100 * (chk.azeite_estimado - chk.azeite_hl) / chk.azeite_hl
print("Erro relativo (nacional): mínimo %.1f%% | máximo %.1f%%" % (chk["erro_%"].min(), chk["erro_%"].max()))
print("(o rendimento vem arredondado a 2 casas, então algum erro é normal)")

# 3.4 A azeitona oleificada nunca deve passar da produção total
comp = nacional[["azeitona_producao_t", "azeitona_oleificada_t"]].dropna()
print("Anos em que oleificada > produção:", list(comp.index[comp.azeitona_oleificada_t > comp.azeitona_producao_t]))

# 3.5 Valores em falta e com asterisco no painel das 7 regiões
print("\nValores em falta (Continente):")
print(longo[longo.regiao_cod.isin(CONTINENTE)].groupby("variavel").valor.apply(lambda s: s.isna().sum()))
print("Valores com asterisco:")
print(longo[longo.flag_asterisco].groupby(["variavel", "ano", "regiao"]).size())

> **Atenção ao período:** a série de **azeite (hl)** que você exportou começa em **2004** (tem uma dimensão extra, "grau de acidez"), enquanto azeitona e rendimento começam em **1995**. Ao juntar tudo, os modelos com azeite terão ~22 anos por região, e os de azeitona ~31. A verificação 3.3 mostra que dá para **reconstruir** o azeite de 1995–2003 a partir de rendimento × azeitona, mas de forma aproximada; só vale a pena se os orientadores concordarem.

## 4. Evolução temporal

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
nacional["azeitona_producao_t"].div(1e3).plot(ax=ax[0], marker="o", ms=3, label="Produção de azeitona")
nacional["azeitona_oleificada_t"].div(1e3).plot(ax=ax[0], marker="o", ms=3, label="Azeitona oleificada")
ax[0].set(title="Portugal: azeitona", ylabel="mil toneladas", xlabel="ano"); ax[0].legend()

nacional["azeite_hl"].div(1e3).plot(ax=ax[1], marker="o", ms=3, color="tab:green")
ax[1].set(title="Portugal: azeite produzido", ylabel="mil hl", xlabel="ano")
plt.tight_layout(); plt.show()

In [ ]:
# Por região (escala logarítmica: o Alentejo é muito maior que as outras)
az = painel.pivot(index="ano", columns="regiao", values="azeitona_producao_t")
az.plot(logy=True, marker="o", ms=3, figsize=(10, 5), title="Produção de azeitona por região agrária (escala log)")
plt.ylabel("toneladas (log)"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.show()

# Peso de cada região no total do Continente
peso = az.div(az.sum(axis=1), axis=0) * 100
peso.plot.area(figsize=(10, 4.5), title="Peso de cada região na produção de azeitona do Continente (%)")
plt.ylabel("%"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.show()
print(peso.loc[[peso.index.min(), peso.index.max()]].round(1))

In [ ]:
# Rendimento: litros de azeite por quintal de azeitona (nacional)
nacional["rendimento_hl_q"].plot(marker="o", ms=3, title="Rendimento em azeite (hl por quintal de azeitona), Portugal")
plt.ylabel("hl/q"); plt.show()

## 5. Alternância produtiva e anos atípicos
A oliveira tende a alternar anos de produção alta e baixa. Isso pode **esconder** qualquer efeito de longo prazo, e é por isso que o plano inclui a produção do ano anterior e testes de robustez. Abaixo usamos a **variação anual em log** (≈ variação percentual), que remove a tendência.

In [ ]:
prod = nacional["azeitona_producao_t"].dropna()
cresc = np.log(prod).diff().dropna()          # variação anual em log
z = (cresc - cresc.mean()) / cresc.std()

fig, ax = plt.subplots(figsize=(10, 4))
cores = ["tab:red" if abs(v) > 1.5 else "tab:blue" for v in z]
ax.bar(cresc.index, cresc.values * 100, color=cores)
ax.axhline(0, color="k", lw=.8)
ax.set(title="Variação anual da produção de azeitona (Portugal) – vermelho: |z| > 1,5", ylabel="% (aprox.)", xlabel="ano")
plt.show()

print("Anos candidatos a 'atípicos' (para o teste de robustez):", list(z.index[z.abs() > 1.5]))

# Autocorrelação das variações: valor negativo nos primeiros lags = alternância
ac = pd.Series({k: cresc.autocorr(k) for k in range(1, 6)})
print("\nAutocorrelação da variação anual (lags 1 a 5):"); print(ac.round(2))

## 6. Correlação cruzada (pronta para a série de oliveiras)
A pergunta do projeto é **em que desfasamento** a série de oliveiras antecipa a produção. Esta função calcula a correlação entre `y` no ano *t* e `x` no ano *t − k*.

Nas **variações anuais em log**, e não nos níveis: duas séries crescentes têm correlação alta só por causa da tendência (correlação espúria). Como há poucos anos, trate os resultados como **exploratórios**, não como prova.

In [ ]:
def correlacao_cruzada(x, y, max_lag=8):
    """Correlação de y(t) com x(t-k), k = 0..max_lag. x e y: Series indexadas por ano."""
    return pd.Series({k: y.corr(x.shift(k)) for k in range(0, max_lag + 1)}, name="corr")

# Exemplo que já funciona: azeitona oleificada (x) vs azeite (y), variações em log
x = np.log(nacional["azeitona_oleificada_t"]).diff()
y = np.log(nacional["azeite_hl"]).diff()
print(correlacao_cruzada(x, y, 4).round(2))   # esperado: forte no lag 0 (é a mesma matéria-prima)

# ---- QUANDO TIVER A SÉRIE DE OLIVEIRAS (ou área de olival) ----
# oliveiras = pd.read_csv("dados_brutos/oliveiras.csv")        # ajuste ao formato real
# s_oliv = oliveiras.set_index("ano")["valor"]
# x = np.log(s_oliv).diff()
# y = np.log(nacional["azeitona_producao_t"]).diff()
# print(correlacao_cruzada(x, y, 8).round(2))

## 7. Próximos passos
- [ ] Confirmar com os orientadores a **fonte da série de oliveiras** (não existe na Base de Dados do INE).
- [ ] Ver na metainformação do INE o significado do `*` e a diferença entre "local de proveniência" e "localização geográfica".
- [ ] Decidir o período comum das séries (azeite só desde 2004).
- [ ] Usar os gráficos das seções 4 e 5 na **apresentação intermédia (E1, até 06/12)**.
- [ ] Depois: regressões com desfasamentos, efeitos fixos por região e random forest/XGBoost com validação cruzada temporal (T3).

Arquivos gerados em `dados_tratados/`: `painel_regiao_ano.csv` (região × ano) e `nacional_ano.csv` (Portugal).